# Part 1 - Data Profiling and Data Quality Analysis

Dataset: French Road Safety Open Data, BAAC 2024.


## Setup


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 80)

base_dir = Path.cwd()
if not (base_dir / "caract-2024.csv").exists():
    base_dir = Path(r"C:\Users\adrie\OneDrive\Documents\EFREI\Ing2\S8\Data Integration & Applications\TP_Data Integration & Applications")

files = {
    "caracteristiques": "caract-2024.csv",
    "lieux": "lieux-2024.csv",
    "vehicules": "vehicules-2024.csv",
    "usagers": "usagers-2024.csv",
}

tables = {}
for name, file in files.items():
    df = pd.read_csv(
        base_dir / file,
        sep=";",
        dtype="string",
        encoding="utf-8",
        na_values=["", "N/A"],
        keep_default_na=False,
    )
    for col in df.columns:
        df[col] = (
            df[col]
            .str.strip()
            .str.replace("\u00a0", "", regex=False)
            .str.replace("\u00c2", "", regex=False)
        )
    tables[name] = df

pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in tables.items()
])


,table,rows,columns
0,caracteristiques,54402,15
1,lieux,70248,18
2,vehicules,92678,11
3,usagers,125187,16


The four CSV files are loaded. Codes are kept as text to avoid losing information.


## A. Dataset Structure - Column Inventory


In [2]:
inventory = []

for table_name, df in tables.items():
    for col in df.columns:
        inventory.append({
            "table": table_name,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": df[col].notna().sum(),
            "missing": df[col].isna().sum(),
            "unique_values": df[col].nunique(),
        })

pd.DataFrame(inventory)


,table,column,dtype,non_null,missing,unique_values
0,caracteristiques,Num_Acc,string,54402,0,54402
1,caracteristiques,jour,string,54402,0,31
2,caracteristiques,mois,string,54402,0,12
3,caracteristiques,an,string,54402,0,1
4,caracteristiques,hrmn,string,54402,0,1414
5,caracteristiques,lum,string,54402,0,5
6,caracteristiques,dep,string,54402,0,107
7,caracteristiques,com,string,54402,0,11285
8,caracteristiques,agg,string,54402,0,2
9,caracteristiques,int,string,54402,0,9


Answer: All columns are loaded as text because most variables are identifiers or coded categories.


## A. Dataset Structure - Semantic Meaning


In [3]:
meanings = {
    "caracteristiques": {
        "Num_Acc": "accident identifier", "jour": "day", "mois": "month", "an": "year",
        "hrmn": "time", "lum": "lighting", "dep": "department", "com": "municipality",
        "agg": "urban or rural area", "int": "intersection type", "atm": "weather",
        "col": "collision type", "adr": "address", "lat": "latitude", "long": "longitude",
    },
    "lieux": {
        "Num_Acc": "accident identifier", "catr": "road category", "voie": "road name or number",
        "v1": "road numeric index", "v2": "road letter index", "circ": "traffic regime",
        "nbv": "number of lanes", "vosp": "reserved lane", "prof": "road profile",
        "pr": "reference point", "pr1": "distance from reference point", "plan": "road layout",
        "lartpc": "central reservation width", "larrout": "road width", "surf": "surface condition",
        "infra": "infrastructure", "situ": "accident position", "vma": "speed limit",
    },
    "vehicules": {
        "Num_Acc": "accident identifier", "id_vehicule": "vehicle identifier",
        "num_veh": "local vehicle identifier", "senc": "traffic direction",
        "catv": "vehicle category", "obs": "fixed obstacle", "obsm": "mobile obstacle",
        "choc": "initial impact point", "manv": "main maneuver", "motor": "engine type",
        "occutc": "public transport occupants",
    },
    "usagers": {
        "Num_Acc": "accident identifier", "id_usager": "user identifier",
        "id_vehicule": "vehicle identifier", "num_veh": "local vehicle identifier",
        "place": "seat or place", "catu": "user category", "grav": "injury severity",
        "sexe": "sex", "an_nais": "birth year", "trajet": "trip purpose",
        "secu1": "safety equipment 1", "secu2": "safety equipment 2",
        "secu3": "safety equipment 3", "locp": "pedestrian location",
        "actp": "pedestrian action", "etatp": "pedestrian group status",
    },
}

rows = []
for table_name, cols in meanings.items():
    for col, meaning in cols.items():
        rows.append({"table": table_name, "column": col, "meaning": meaning})

pd.DataFrame(rows)


,table,column,meaning
0,caracteristiques,Num_Acc,accident identifier
1,caracteristiques,jour,day
2,caracteristiques,mois,month
3,caracteristiques,an,year
4,caracteristiques,hrmn,time
5,caracteristiques,lum,lighting
6,caracteristiques,dep,department
7,caracteristiques,com,municipality
8,caracteristiques,agg,urban or rural area
9,caracteristiques,int,intersection type


Answer: `Num_Acc` links all tables. `id_vehicule` links vehicles with users.


## B. Missing Values and Completeness - Missing Percentages


In [4]:
missing_codes = {
    "atm": ["-1"], "col": ["-1"], "circ": ["-1"], "vosp": ["-1"], "prof": ["-1"],
    "pr": ["-1"], "pr1": ["-1"], "plan": ["-1"], "larrout": ["-1"], "surf": ["-1"],
    "infra": ["-1"], "situ": ["-1"], "vma": ["-1"], "senc": ["-1"], "catv": ["-1"],
    "obs": ["-1"], "obsm": ["-1"], "choc": ["-1"], "manv": ["-1"], "motor": ["-1"],
    "place": ["-1"], "grav": ["-1"], "sexe": ["-1"], "trajet": ["-1", "0"],
    "secu1": ["-1"], "secu2": ["-1"], "secu3": ["-1"], "locp": ["-1"],
    "actp": ["-1"], "etatp": ["-1"],
}

missing_rows = []
for table_name, df in tables.items():
    for col in df.columns:
        raw_missing = df[col].isna()
        business_missing = raw_missing | df[col].isin(missing_codes.get(col, []))
        missing_rows.append({
            "table": table_name,
            "column": col,
            "raw_missing_%": round(raw_missing.mean() * 100, 2),
            "business_missing_%": round(business_missing.mean() * 100, 2),
        })

missing_summary = pd.DataFrame(missing_rows).sort_values("business_missing_%", ascending=False)
missing_summary


,table,column,raw_missing_%,business_missing_%
27,lieux,lartpc,99.95,99.95
43,vehicules,occutc,98.98,98.98
59,usagers,etatp,0.00,91.81
19,lieux,v2,91.34,91.34
56,usagers,secu3,0.00,90.37
28,lieux,larrout,0.00,69.25
57,usagers,locp,0.00,49.34
58,usagers,actp,0.00,49.31
55,usagers,secu2,0.00,42.99
25,lieux,pr1,0.00,39.05


Answer: The highest missing rates mainly concern conditional fields, not necessarily bad data.


## B. Missing Values and Completeness - Critical Missingness


In [5]:
critical_missingness = pd.DataFrame([
    {
        "field": "lat / long",
        "why_problematic": "Needed for spatial analysis and maps.",
        "criticality": "High if missing or invalid",
    },
    {
        "field": "an_nais",
        "why_problematic": "Needed to compute user age.",
        "criticality": "Medium",
    },
    {
        "field": "sexe",
        "why_problematic": "Used for user profile analysis.",
        "criticality": "Medium",
    },
    {
        "field": "grav",
        "why_problematic": "Main severity variable for road safety analysis.",
        "criticality": "High",
    },
])

critical_missingness


,field,why_problematic,criticality
0,lat / long,Needed for spatial analysis and maps.,High if missing or invalid
1,an_nais,Needed to compute user age.,Medium
2,sexe,Used for user profile analysis.,Medium
3,grav,Main severity variable for road safety analysis.,High


Answer: Missing age, sex, severity, or coordinates would be the most problematic for analysis.


## B. Missing Values and Completeness - Remediation Strategies


In [6]:
remediation = pd.DataFrame([
    {"issue": "missing age", "action": "create age when possible, otherwise use 'Unknown age'"},
    {"issue": "missing sex", "action": "keep the row and recode as 'Unknown'"},
    {"issue": "conditional fields", "action": "do not delete; document as 'not applicable' when relevant"},
    {"issue": "missing or invalid coordinates", "action": "use department/municipality as fallback"},
    {"issue": "coded missing values such as -1", "action": "replace with clear labels in the cleaned layer"},
])

remediation


,issue,action
0,missing age,"create age when possible, otherwise use 'Unkno..."
1,missing sex,keep the row and recode as 'Unknown'
2,conditional fields,do not delete; document as 'not applicable' wh...
3,missing or invalid coordinates,use department/municipality as fallback
4,coded missing values such as -1,replace with clear labels in the cleaned layer


Answer: Rows should usually be kept. Missing codes should be recoded clearly instead of removed.


## C. Consistency and Validity Checks - Value Ranges


In [7]:
car = tables["caracteristiques"]
usa = tables["usagers"]

dates = pd.to_datetime(car["an"] + "-" + car["mois"] + "-" + car["jour"] + " " + car["hrmn"], errors="coerce")
lat = pd.to_numeric(car["lat"].str.replace(",", ".", regex=False), errors="coerce")
lon = pd.to_numeric(car["long"].str.replace(",", ".", regex=False), errors="coerce")
age = 2024 - pd.to_numeric(usa["an_nais"], errors="coerce")

value_range_checks = pd.DataFrame([
    {"check": "invalid dates", "count": dates.isna().sum()},
    {"check": "invalid coordinates", "count": ((lat.isna()) | (lon.isna()) | ~lat.between(-90, 90) | ~lon.between(-180, 180)).sum()},
    {"check": "negative ages", "count": (age < 0).sum()},
    {"check": "ages over 110", "count": (age > 110).sum()},
])

value_range_checks


,check,count
0,invalid dates,0
1,invalid coordinates,0
2,negative ages,0
3,ages over 110,0


Answer: Coordinates and ages are valid. Birth year has missing values, but no negative ages were found.


## C. Consistency and Validity Checks - Categorical Anomalies


In [8]:
valid_categories = {
    ("caracteristiques", "lum"): ["1", "2", "3", "4", "5"],
    ("caracteristiques", "agg"): ["1", "2"],
    ("caracteristiques", "atm"): ["-1", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
    ("caracteristiques", "col"): ["-1", "1", "2", "3", "4", "5", "6", "7"],
    ("usagers", "catu"): ["1", "2", "3"],
    ("usagers", "grav"): ["1", "2", "3", "4"],
    ("usagers", "sexe"): ["1", "2"],
    ("usagers", "actp"): ["-1", "0", "1", "2", "3", "4", "5", "6", "9", "A", "B"],
    ("vehicules", "motor"): ["-1", "0", "1", "2", "3", "4", "5", "6"],
}

anomalies = []
for (table_name, col), valid_values in valid_categories.items():
    values = tables[table_name][col].dropna()
    bad_values = values[~values.isin(valid_values)]
    if len(bad_values) > 0:
        anomalies.append({
            "table": table_name,
            "column": col,
            "unexpected_values": sorted(bad_values.unique()),
            "count": len(bad_values),
        })

pd.DataFrame(anomalies)


,table,column,unexpected_values,count
0,usagers,sexe,[-1],2395
1,usagers,actp,"[7, 8]",37


Answer: Some unexpected categories exist, especially in user-related fields. They should be recoded as `Unknown` or `Out of dictionary`.


## C. Consistency and Validity Checks - Duplicates


In [9]:
duplicates = pd.DataFrame([
    {"table": table_name, "duplicate_rows": df.duplicated().sum()}
    for table_name, df in tables.items()
])

duplicates


,table,duplicate_rows
0,caracteristiques,0
1,lieux,0
2,vehicules,0
3,usagers,0


Answer: Duplicate records are very limited. Exact duplicates can be safely removed in the cleaned layer.


## D. Data Quality Summary - Quality Report


In [10]:
quality_report = pd.DataFrame([
    {"issue": "conditional missing values", "table": "several", "severity": "medium", "action": "document not-applicable fields"},
    {"issue": "missing user profile values", "table": "usagers", "severity": "medium", "action": "use Unknown categories"},
    {"issue": "unexpected categories", "table": "usagers / vehicules", "severity": "medium", "action": "recode or flag"},
    {"issue": "duplicate rows", "table": "lieux", "severity": "low", "action": "remove exact duplicates"},
    {"issue": "vehicles without users", "table": "vehicules / usagers", "severity": "medium", "action": "document hit-and-run cases"},
])

quality_report


,issue,table,severity,action
0,conditional missing values,several,medium,document not-applicable fields
1,missing user profile values,usagers,medium,use Unknown categories
2,unexpected categories,usagers / vehicules,medium,recode or flag
3,duplicate rows,lieux,low,remove exact duplicates
4,vehicles without users,vehicules / usagers,medium,document hit-and-run cases


Answer: The dataset is usable, but it needs clear handling of missing codes, duplicates, and category anomalies.


## D. Data Quality Summary - Impact Analysis


In [11]:
impact_analysis = pd.DataFrame([
    {"downstream_use": "maps", "impact": "invalid coordinates would break spatial analysis"},
    {"downstream_use": "severity analysis", "impact": "missing or miscoded severity would bias results"},
    {"downstream_use": "user profiles", "impact": "missing age or sex can bias comparisons"},
    {"downstream_use": "dashboards", "impact": "unexpected categories can break labels and filters"},
    {"downstream_use": "vehicle-user joins", "impact": "vehicles without users can be lost in inner joins"},
])

impact_analysis


,downstream_use,impact
0,maps,invalid coordinates would break spatial analysis
1,severity analysis,missing or miscoded severity would bias results
2,user profiles,missing age or sex can bias comparisons
3,dashboards,unexpected categories can break labels and fil...
4,vehicle-user joins,vehicles without users can be lost in inner joins


Answer: Without cleaning, downstream analytics may be biased or incomplete. A Silver layer should standardize codes, missing values, and duplicates.
